# MISC

In [1]:
import os, sys
import pandas as pd
import numpy as np
from datetime import datetime
import json
import glob

notebook_dir = os.getcwd()  # Gets test folder path
project_dir = os.path.dirname(notebook_dir)  # Gets Project folder path
sys.path.append(project_dir)

from subproblems import *
from utils import *

# JSON configuration

In [3]:
data = {
    "config name": "config_testing_1",
    "project_name": "new collection testing",
    "sce_start": 1,
    "sce_end": 10,
    "last_processed_index": 0,
    "iter_save": 50,
    "number of decisions": 4,
    "is_csv": False,
    "save_csv": True,
    "ADMM_ver": "D:/Jacky/Python/ADMM_P2P_Python/main/ADMM_fcn_TC1_LP_PrioGO.jl",
    "testdataset": "D:/Jacky/Julia-vscode/ADMM_P2P/Output/TC1_LP_PrioGO_fix_test_20",
    "primal_pred": "D:/Jacky/Python/JKFYP/GRU/primal_pred_PL_FT5.npy",
    "dual_pred": "D:/Jacky/Python/JKFYP/GRU/dual_pred_PL_FT1.npy",
}

filename = f"{data['config name']}.json"
with open(filename, 'w') as json_file:
    json.dump(data, json_file, indent=4) # Using indent for human-readable output

In [11]:
data = {
    "config name": "config_training_1",
    "project_name": "LP_PrioGO_test_20",
    "sce_start": 1,
    "sce_end": 20,
    "last_processed_index": 0,
    "iter_save": 50,
    "number of decisions": 4,
    "save_csv": False,
    "ADMM_ver": "D:/Jacky/Python/ADMM_P2P_Python/main/ADMM_fcn_TC1_LP_PrioGO.jl",
}

filename = f"{data['config name']}.json"
with open(filename, 'w') as json_file:
    json.dump(data, json_file, indent=4) # Using indent for human-readable output

In [5]:
np.load(r"D:\Jacky\Data Output\ADMM_P2P\New\TC1_LP_PrioGO_fix_test_20_New_ver2\optimal_iter\conv_iter.npz").shape

(20,)

# Pre processing folder_path

In [2]:
# General Info
folder_path = r"D:\Jacky\Data Output\ADMM_P2P\New\TC1_LP_PrioGO_fix_test_20_New_ver2"
with open(f'{folder_path}/config.json', 'r') as file:
    config = json.load(file)
total_sce = config["sce_end"] - config["sce_start"] + 1
n_sce, start_sce_save, end_sce_save = 5, config["sce_start"], config["sce_end"]
num_user = 32
hour = 48
n_bus, n_branch = 33, 32
tot_iter_save = config["iter_save"] + 1
max_timestep = 5000
n_dec = config["number of decisions"]

FileNotFoundError: [Errno 2] No such file or directory: 'D:\\Jacky\\Data Output\\ADMM_P2P\\New\\TC1_LP_PrioGO_fix_test_20_New_ver2/config.json'

In [ ]:
# CSV - combine and infeasible filter
if config["save_csv"]:
    all_dual = None
    all_primal = None
    all_infeasible = []

    # 1. Read all infeasible scenario indices
    data_directory = f"{folder_path}/infeasible_sce/"
    # Search for files matching the pattern
    infeasible_files = glob.glob(os.path.join(data_directory, "infeasible_sce_*to*sce.csv"))

    for file_path in infeasible_files:
        print(os.path.basename(file_path))
        df = pd.read_csv(file_path)
        infeasible_mat = df.values
        
        # Store all infeasible indices
        for inf_sce in infeasible_mat.flatten():
            all_infeasible.append(inf_sce)

    # Unique and sorted list
    all_infeasible = sorted(list(set(all_infeasible)))

    # 2. Read main CSV files
    i = start_sce_save
    while i <= end_sce_save - 4:
        print(i)
        first = i
        last = i + 4
        
        path_dv = f"{folder_path}/DecisionVariable/dual_{first}to{last}sce.csv"
        path_pv = f"{folder_path}/DecisionVariable/primal_{first}to{last}sce.csv"
        
        # Read CSVs
        df_dv = pd.read_csv(path_dv)
        df_pv = pd.read_csv(path_pv)
        
        # Convert to NumPy and Reshape
        # Note: Julia reshapes column-major (F), Python is row-major (C) by default.
        # To match Julia's reshape(5, 192, 32, 51), we use order='F'
        lambda_2dnew = df_dv.values
        lambda_4d = lambda_2dnew.reshape((5, n_dec * hour, num_user, tot_iter_save), order='F')
        
        pout_2dnew = df_pv.values
        pout_4d = pout_2dnew.reshape((5, n_dec * hour, num_user, tot_iter_save), order='F')
        
        # Concatenate along the first dimension (axis=0)
        if all_dual is None:
            all_dual = lambda_4d
            all_primal = pout_4d
        else:
            all_dual = np.concatenate((all_dual, lambda_4d), axis=0)
            all_primal = np.concatenate((all_primal, pout_4d), axis=0)
        
        i += 5

    # 3. Adjust indices for slicing
    if start_sce_save != 1:
        # Converting to Python's 0-based indexing and adjusting by start offset
        all_infeasible = [int(x - start_sce_save) for x in all_infeasible]
    else:
        # Just shift to 0-based
        all_infeasible = [int(x - 1) for x in all_infeasible]

    # 4. Remove infeasible indices
    # Create a mask of indices to keep
    total_scenarios = all_dual.shape[0]
    remaining_indices = [idx for idx in range(total_scenarios) if idx not in all_infeasible]

    new_all_dual = all_dual[remaining_indices, :, :, :]
    new_all_primal = all_primal[remaining_indices, :, :, :]

    # 5. Reshape back to 2D and Save
    # To match Julia's CSV output, we use order='F' during the flatten
    lambda_2d = new_all_dual.reshape(-1, all_dual.shape[3], order='F')
    pout_2d = new_all_primal.reshape(-1, all_primal.shape[3], order='F')

    dv = pd.DataFrame(lambda_2d)
    pv = pd.DataFrame(pout_2d)

    out_path_dual = f"{folder_path}/dual_{start_sce_save}to{end_sce_save}sce_feasible.csv"
    out_path_primal = f"{folder_path}/primal_{start_sce_save}to{end_sce_save}sce_feasible.csv"

    dv.to_csv(out_path_dual, index=False)
    pv.to_csv(out_path_primal, index=False)

    print(f"Total feasible scenario = {total_sce} - {len(all_infeasible)} = {total_sce - len(all_infeasible)}")

In [ ]:
# NPZ - infeasible filter

In [ ]:
# NPZ - GRU format
primal = []
dual = []
decision_var = []

## for the sack of reshape and memory issue
primal_temp = np.load(f"{folder_path}/DecisionVariable/primal.npz")
dual_temp = np.load(f"{folder_path}/DecisionVariable/dual.npz")
decision_var_temp = np.load(f"{folder_path}/DecisionVariable/decision_var.npz")

for i in range(int(total_sce/n_sce)):
    primal.append(primal_temp[i*n_sce*num_user*n_dec*hour:(i+1)*n_sce*num_user*n_dec*hour,:].reshape(n_sce, -1, num_user, tot_iter_save, order="F"))
    dual.append(dual_temp[i*n_sce*num_user*n_dec*hour:(i+1)*n_sce*num_user*n_dec*hour,:].reshape(n_sce, -1, num_user, tot_iter_save, order="F"))
    # decision_var.append(decision_var_temp[i*n_sce*num_user*8*hour:(i+1)*n_sce*num_user*8*hour,:].reshape(n_sce, -1, num_user, tot_iter_save, order="F"))
primal, dual = np.concatenate(primal, 0), np.concatenate(dual, 0)
# decision_var = np.concatenate(decision_var, 0)

np.save("primalGRU", primal)
np.save("dualGRU", dual)
np.save("decisionGRU", decision_var)

del primal, dual, primal_temp, dual_temp, decision_var, decision_var_temp

In [ ]:
if not config["save_csv"]:
    all_dual = None
    all_primal = None
    all_infeasible = []

    # 1. Read all infeasible scenario indices
    data_directory = f"{folder_path}/infeasible_sce/"
    # Search for files matching the pattern
    infeasible_files = glob.glob(os.path.join(data_directory, "infeasible_sce_*to*sce.csv"))

    for file_path in infeasible_files:
        print(os.path.basename(file_path))
        df = pd.read_csv(file_path)
        infeasible_mat = df.values
        
        # Store all infeasible indices
        for inf_sce in infeasible_mat.flatten():
            all_infeasible.append(inf_sce)

    # Unique and sorted list
    all_infeasible = sorted(list(set(all_infeasible)))

    # 2. Read main NPZ files
        path_dv = f"{folder_path}/DecisionVariable/dual.npz"
        path_pv = f"{folder_path}/DecisionVariable/primal.npz"
        
        # Read npz
        df_dv = np.load(path_dv)
        df_pv = np.load(path_pv)
        
        # Convert to NumPy and Reshape
        # Note: Julia reshapes column-major (F), Python is row-major (C) by default.
        # To match Julia's reshape(5, 192, 32, 51), we use order='F'
        lambda_2dnew = df_dv.values
        lambda_4d = lambda_2dnew.reshape((5, n_dec * hour, num_user, tot_iter_save), order='F')
        
        pout_2dnew = df_pv.values
        pout_4d = pout_2dnew.reshape((5, n_dec * hour, num_user, tot_iter_save), order='F')
        
        # Concatenate along the first dimension (axis=0)
        if all_dual is None:
            all_dual = lambda_4d
            all_primal = pout_4d
        else:
            all_dual = np.concatenate((all_dual, lambda_4d), axis=0)
            all_primal = np.concatenate((all_primal, pout_4d), axis=0)
        
        i += 5

    # 3. Adjust indices for slicing
    if start_sce_save != 1:
        # Converting to Python's 0-based indexing and adjusting by start offset
        all_infeasible = [int(x - start_sce_save) for x in all_infeasible]
    else:
        # Just shift to 0-based
        all_infeasible = [int(x - 1) for x in all_infeasible]

    # 4. Remove infeasible indices
    # Create a mask of indices to keep
    total_scenarios = all_dual.shape[0]
    remaining_indices = [idx for idx in range(total_scenarios) if idx not in all_infeasible]

    new_all_dual = all_dual[remaining_indices, :, :, :]
    new_all_primal = all_primal[remaining_indices, :, :, :]

    # 5. Reshape back to 2D and Save
    # To match Julia's CSV output, we use order='F' during the flatten
    lambda_2d = new_all_dual.reshape(-1, all_dual.shape[3], order='F')
    pout_2d = new_all_primal.reshape(-1, all_primal.shape[3], order='F')

    dv = pd.DataFrame(lambda_2d)
    pv = pd.DataFrame(pout_2d)

    out_path_dual = f"{folder_path}/dual_{start_sce_save}to{end_sce_save}sce_feasible.csv"
    out_path_primal = f"{folder_path}/primal_{start_sce_save}to{end_sce_save}sce_feasible.csv"

    dv.to_csv(out_path_dual, index=False)
    pv.to_csv(out_path_primal, index=False)

    print(f"Total feasible scenario = {total_sce} - {len(all_infeasible)} = {total_sce - len(all_infeasible)}")